# 路由交换分析


In [ ]:
from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

os.environ["CUDA_VISIBLE_DEVICES"] = ""
_pkg_dir = Path(__file__).parent.resolve()
if str(_pkg_dir) not in sys.path:
    sys.path.insert(0, str(_pkg_dir))

_core_dir = _pkg_dir.parent / "core"
if str(_core_dir) not in sys.path:
    sys.path.insert(0, str(_core_dir))
from pipeline import Config, load_model_and_tokenizer

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

cfg = Config()
model, tokenizer, device = load_model_and_tokenizer(
    cfg.model_path, device=cfg.device,
    torch_dtype=getattr(torch, cfg.torch_dtype, torch.bfloat16),
)

prompt = "def foo(x): return x + x * x + bar(x)"
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32)
input_ids = inputs.input_ids.to(device)
T = input_ids.shape[1]

# 注册所有 MoE 层的钩子
moe_inputs = {}
moe_outputs = {}
handles = []

for l in range(cfg.num_hidden_layers):
    layer = model.model.layers[l]
    if hasattr(layer.mlp, 'experts'):   # MoE 层（非稠密层）
        def make_hook(layer_idx):
            def hook(m, i, o):
                # BailingMoeV2SparseMoeBlock.forward 返回 (y, (router, topk))
                # o 是完整的输出元组
                if isinstance(o, tuple) and len(o) == 2:
                    moe_inputs[layer_idx] = i[0].detach().cpu()
                    moe_outputs[layer_idx] = o[0].detach().cpu()
                    if isinstance(o[1], tuple) and len(o[1]) == 2:
                        moe_outputs[f'{layer_idx}_router'] = o[1][0].detach().cpu()
                        moe_outputs[f'{layer_idx}_topk'] = o[1][1].detach().cpu()
            return hook
        h = layer.mlp.register_forward_hook(make_hook(l))
        handles.append(h)

with torch.no_grad():
    outputs = model(input_ids=input_ids, output_router_logits=True, use_cache=False, return_dict=True)

for h in handles:
    h.remove()

# 也获取 MTP 的 MoE 块
mtp_moe = model.model.layers[-1].mlp

# 识别 MoE 层（跳过稠密层 - they don't have router_logits in all_router)
all_router = outputs.router_logits
num_moe_layers = len(all_router) - 1   # MoE Decoder 层
# all_router[-1] 是 MTP，all_router[-2] 是最后一个 Decoder MoE, etc.

# 映射：Decoder 层索引对应哪个 all_router 项？
# 第 0 层是稠密层 → no router logits
# 第 1-18 层是 MoE → all_router[0..17]
# 第 19 层是最后一个 Decoder MoE → all_router[18]
# MTP→all_router[19]

print(f"\nPrompt: {prompt}")
print(f"Tokens: {T}")
print(f"Total MoE decoder layers: {num_moe_layers}")
print(f"Captured MoE inputs: {[k for k in moe_inputs.keys() if isinstance(k, int)]}")

# 对每个 MoE Decoder 层，测试 MTP 路由 → Decoder experts
# 注意：model.layers[0..18] 是 Decoder，model.layers[19] 是 MTP
# 跳过第 0 层（稠密）和第 19 层（MTP）
# 只分析真正的 Decoder MoE 层: 1..18
decoder_moe_indices = [l for l in range(1, cfg.num_hidden_layers - 1) if l in moe_inputs]
print(f"\nTrue decoder MoE layers: {len(decoder_moe_indices)} entries: {decoder_moe_indices}")

print(f"\n{'='*80}")
print(f"Across ALL MoE layers: MTP routing → Decoder experts")
print(f"{'='*80}")
print(f"{'Layer':<8} {'overlap/8':<12} {'Cos(MTP→Dec)':<15} {'Cos(MTP→MTP)':<15}")

results = []
for decoder_layer_idx in decoder_moe_indices:
    layer = model.model.layers[decoder_layer_idx]

    moe_in = moe_inputs[decoder_layer_idx]
    dec_moe_out = moe_outputs[decoder_layer_idx]   # Decoder 的实际输出
    dec_router_logits = moe_outputs[f'{decoder_layer_idx}_router']
    dec_topk_idx = moe_outputs[f'{decoder_layer_idx}_topk']
    decoder_moe = layer.mlp

    # 获取 MTP 在此层输入上的路由
    with torch.no_grad():
        mtp_gate_out = mtp_moe.gate(moe_in)
    mtp_topk_idx = mtp_gate_out[0]
    mtp_topk_wgt = mtp_gate_out[1]

    # 索引重叠
    overlap_total = 0
    for t in range(T):
        d_set = set(dec_topk_idx[0, t].tolist())
        m_set = set(mtp_topk_idx[t].tolist())   # gate 返回 [T, K]
        overlap_total += len(d_set & m_set)
    avg_overlap = overlap_total / T

    # 用 MTP 路由计算 MoE 输出 → decoder's experts
    bsz = moe_in.shape[0]
    D = moe_in.shape[-1]
    flat_in = moe_in.view(-1, D)
    with torch.no_grad():
        mtp2dec_y = decoder_moe.moe_infer(flat_in, mtp_topk_idx, mtp_topk_wgt).view(bsz, T, D)
        if decoder_moe.shared_experts is not None:
            mtp2dec_y = mtp2dec_y + decoder_moe.shared_experts(moe_in)

        mtp_mtp_y = mtp_moe.moe_infer(flat_in, mtp_topk_idx, mtp_topk_wgt).view(bsz, T, D)
        if mtp_moe.shared_experts is not None:
            mtp_mtp_y = mtp_mtp_y + mtp_moe.shared_experts(moe_in)

    avg_cos_md = sum(F.cosine_similarity(mtp2dec_y[0, t].unsqueeze(0).float(), dec_moe_out[0, t].unsqueeze(0).float()).item() for t in range(T)) / T
    avg_cos_mm = sum(F.cosine_similarity(mtp_mtp_y[0, t].unsqueeze(0).float(), dec_moe_out[0, t].unsqueeze(0).float()).item() for t in range(T)) / T

    layer_type = "dense" if decoder_layer_idx < 1 else "MoE"
    print(f"L{decoder_layer_idx:<5} {layer_type:<10} {avg_overlap:<8.2f}/8     {avg_cos_md:<15.4f} {avg_cos_mm:<15.4f}")
    results.append({
        'layer': decoder_layer_idx,
        'overlap': avg_overlap,
        'cos_md': avg_cos_md,
        'cos_mm': avg_cos_mm,
    })

# 整体平均
if results:
    avg_overlap_all = sum(r['overlap'] for r in results) / len(results)
    avg_cos_md_all = sum(r['cos_md'] for r in results) / len(results)
    avg_cos_mm_all = sum(r['cos_mm'] for r in results) / len(results)
    print(f"{'ALL DEC':<8} {avg_overlap_all:<8.2f}/8     {avg_cos_md_all:<15.4f} {avg_cos_mm_all:<15.4f}")

# 检查交换后最终输出预测什么 token？
print(f"\n{'='*80}")
print(f"End-to-end: Token prediction after replacing routing in all layers")
print(f"{'='*80}")

# 获取 Decoder 的实际输出序列 (all layers)
# 需要重建如果每层都用 MTP 路由时 lm_head 的输出
# 用 MTP 路由 + Decoder 专家权重。
#
# 这很复杂，因为不能轻易干预所有层。
# 改为模拟：
# 从第一层输入（embeddings）开始
# 对每层计算 MoE 输出 using MTP's routing → decoder's experts
# 通过剩余部分（attention, residual, norm）
#
# 但这需要重新实现完整前向传播...
# 
# 简化：仅对最后一层, we verified Cos=0.92.
# 最终 lm_head 输出 Cos=0.97（来自第7节）.
# 检查 token 预测是否改变。

print(f"\n--- Token prediction stability (last decoder layer only) ---")
last_layer_idx = decoder_moe_indices[-1]   # 第 18 层，真正的最后一个 Decoder MoE 层
last_input = moe_inputs[last_layer_idx]
last_decoder = model.model.layers[last_layer_idx]
last_decoder_moe = last_decoder.mlp

with torch.no_grad():
    # 获取 MTP 的路由
    mtp_gate = mtp_moe.gate(last_input)
    mtp_idx = mtp_gate[0]
    mtp_wgt = mtp_gate[1]
    
    # 用 MTP 的路由计算输出 → decoder's experts
    bsz, T_l, D_l = last_input.shape
    flat = last_input.view(-1, D_l)
    moe_out_swapped = last_decoder_moe.moe_infer(flat, mtp_idx, mtp_wgt).view(bsz, T_l, D_l)
    if last_decoder_moe.shared_experts is not None:
        moe_out_swapped = moe_out_swapped + last_decoder_moe.shared_experts(last_input)
    
    # Decoder 层输出 = 残差 + moe_out
    # 残差 = last_input（attention + MoE 之前）
    # 但实际上 MoE 输入在 attention 之后, so the residual is different.
    # 直接使用捕获的输出。
    
    # 获取实际最终隐藏状态
    raw_out = model.model(input_ids=input_ids, use_cache=False, return_dict=True)
    actual_final = raw_out.last_hidden_state    # [1, T, D]
    
    # 获取实际 logits
    actual_logits = model.lm_head(actual_final).float()

# 模拟最后一层的交换后 logits
# 最后一层 Decoder 的实际前向路径：
#   hidden_in = moe_input（已捕获）
#   →attention→residual→layernorm→MoE(input)→+residual→norm→lm_head
# 我们有 MoE 输入，可以计算交换后的 MoE 输出
# 但重建完整路径是复杂的。

# 改为验证关键指标: does swapping routing change the TOP-1 token?
# 可以通过逐位置效果来检查。
print(f"\n  Token prediction at each position (original vs MTP-routed last layer):")
for t in range(T):
    # 获取 Decoder 的 MoE 输出对最终 logits 的影响
    actual_moe_out = moe_outputs[last_layer_idx][0, t]    # [D]
    swapped_moe_out = moe_out_swapped[0, t]    # [D]
    
    # 经过 residual→norm→lm_head
    # 残差 = last_layer_input（含 attention）
    # 可以近似：如果 MoE 输出变化 delta, the final hidden changes by norm(delta)
    # 但实际上变化被 layer_norm + lm_head 衰减
    
    cos = F.cosine_similarity(actual_moe_out.unsqueeze(0).float(), swapped_moe_out.unsqueeze(0).float()).item()
    l2 = (actual_moe_out.float() - swapped_moe_out.float()).norm().item()
    print(f"  pos {t:2d}: MoE output Cos={cos:.4f}, L2={l2:.4f}")

print(f"\n{'='*80}")
print(f"SUMMARY")
print(f"{'='*80}")
print(f"1. MTP's routing indices overlap with decoder's: ~{avg_overlap_all:.2f}/8 (near zero)")
print(f"2. But MTP routing → Decoder experts MoE output: avg Cos={avg_cos_md_all:.4f}")
print(f"3. This holds across ALL decoder layers")
print(f"4. Conclusion: Expert selection is highly redundant.")
print(f"   Different subsets of 8 experts produce ~92% similar outputs.")
print(f"   MTP's routing IS a valid proxy for decoder's routing.")

print("\nDone.")

---
## layer_test（分层测试）


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

cfg = Config()
model, tokenizer, device = load_model_and_tokenizer(
    cfg.model_path, device=cfg.device,
    torch_dtype=getattr(torch, cfg.torch_dtype, torch.bfloat16),
)

prompt = "def foo(x): return x + x * x + bar(x)"
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32)
input_ids = inputs.input_ids.to(device)
T = input_ids.shape[1]

mtp_moe = model.model.layers[-1].mlp

# 识别所有 Decoder MoE 层
all_moe = [(idx, model.model.layers[idx].mlp) for idx in range(cfg.num_hidden_layers - 1)
           if hasattr(model.model.layers[idx].mlp, 'experts') and len(model.model.layers[idx].mlp.experts) > 0]

print(f"Total MoE layers: {len(all_moe)}")

# 基线
with torch.no_grad():
    base_out = model(input_ids=input_ids, use_cache=False, return_dict=True)
base_logits = base_out.logits.float()
base_tokens = base_logits.argmax(dim=-1)

def run_with_swap(layer_indices, label):
    handles = []
    for idx, moe in all_moe:
        if idx in layer_indices:
            def make_hook(ref):
                def hook(m, i, o):
                    with torch.no_grad():
                        moe_in = i[0]
                        mtp_gate = ref.gate(moe_in)
                        flat = moe_in.view(-1, moe_in.shape[-1])
                        swapped = m.moe_infer(flat, mtp_gate[0], mtp_gate[1]).view_as(moe_in)
                        if m.shared_experts is not None:
                            swapped = swapped + m.shared_experts(moe_in)
                    return (swapped, o[1])
                return hook
            handles.append(moe.register_forward_hook(make_hook(mtp_moe)))

    with torch.no_grad():
        out = model(input_ids=input_ids, use_cache=False, return_dict=True)

    for h in handles:
        h.remove()

    swap_logits = out.logits.float()
    swap_tokens = swap_logits.argmax(dim=-1)
    match = (base_tokens == swap_tokens).sum().item()

    # Logits 余弦相似度
    logit_cos = 0
    for t in range(T):
        logit_cos += F.cosine_similarity(base_logits[0,t].unsqueeze(0).float(), swap_logits[0,t].unsqueeze(0).float()).item()
    logit_cos /= T

    # 坍缩检测：统计唯一 token 数 in swapped output
    unique_tokens = swap_tokens.unique().numel()
    top_token = swap_tokens.mode().values.item()
    top_token_str = tokenizer.decode(top_token)

    print(f"\n{label}:")
    print(f"  Logit Cos={logit_cos:.4f}, Token match={match}/{T} ({match/T*100:.1f}%)")
    print(f"  Unique tokens in output={unique_tokens}, most common='{top_token_str}'")

    for t in range(T):
        orig = tokenizer.decode(base_tokens[0,t].item())
        swp = tokenizer.decode(swap_tokens[0,t].item())
        tok = tokenizer.decode(input_ids[0,t].item())
        m = "OK" if base_tokens[0,t].item() == swap_tokens[0,t].item() else "X"
        print(f"    pos {t:2d} {tok:<12} {orig:<15} -> {swp:<15} {m}")

    return match, logit_cos, unique_tokens

# 测试配置
configs = [
    ("Shallow (L1-L4)",    [idx for idx, _ in all_moe if idx <= 3]),    # 索引 0-3 → layers 1-4
    ("Middle (L5-L13)",    [idx for idx, _ in all_moe if 4 <= idx <= 12]),
    ("Deep (L14-L18)",     [idx for idx, _ in all_moe if idx >= 13]),
    ("Shallow+Deep",       [idx for idx, _ in all_moe if idx <= 3 or idx >= 13]),
    ("Single L1",          [all_moe[0][0]]),
    ("Single L9",          [all_moe[8][0]]),
    ("Single L18",         [all_moe[-1][0]]),
    ("ALL layers",         [idx for idx, _ in all_moe]),
]

results = []
for label, layers in configs:
    m, lc, ut = run_with_swap(set(layers), label)
    results.append((label, m, lc, ut))

print(f"\n{'='*90}")
print("SUMMARY")
print(f"{'='*90}")
print(f"{'Config':<20} {'Match':<15} {'LogitCos':<12} {'UniqueTok':<12} {'Collapse?':<12}")
print(f"{'------':<20} {'-----':<15} {'--------':<12} {'--------':<12} {'--------':<12}")

for label, match, lc, ut in results:
    collapse = "YES" if ut <= 2 else "no"
    print(f"{label:<20} {match}/{T} ({match/T*100:.1f}%)  {lc:<12.4f} {ut:<12} {collapse:<12}")

print("\nDone.")

---
## collapse_test（坍缩测试）


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

cfg = Config()
model, tokenizer, device = load_model_and_tokenizer(
    cfg.model_path, device=cfg.device,
    torch_dtype=getattr(torch, cfg.torch_dtype, torch.bfloat16),
)

prompt = "def foo(x): return x + x * x + bar(x)"
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32)
input_ids = inputs.input_ids.to(device)
T = input_ids.shape[1]

mtp_moe = model.model.layers[-1].mlp

# 找到 Decoder MoE 层
decoder_moe = [(idx, model.model.layers[idx].mlp) for idx in range(1, cfg.num_hidden_layers - 1)
               if hasattr(model.model.layers[idx].mlp, 'experts')]

print(f"Prompt: {prompt}")
print(f"Tokens: {T}, MoE layers: {len(decoder_moe)}")

# 注册捕获所有层 MoE 输入的钩子
moe_inputs = {}
moe_outputs = {}
handles = []

for idx, moe in decoder_moe:
    def mk_hooks(i_idx):
        def pre(m, inp):
            moe_inputs[i_idx] = inp[0].detach().cpu()
        def post(m, inp, out):
            moe_outputs[i_idx] = out[0].detach().cpu()
        handles.append(moe.register_forward_pre_hook(pre))
        handles.append(moe.register_forward_hook(post))
    mk_hooks(idx)

with torch.no_grad():
    outputs = model(input_ids=input_ids, output_router_logits=True,
                    output_hidden_states=True, use_cache=False, return_dict=True)
for h in handles:
    h.remove()

# 测试1：渐进式坍缩 - swap layers one by one and track final logit
print(f"\n{'='*80}")
print("Progressive collapse: swap layers one by one, track output")
print(f"{'='*80}")

# 从最后一层到第一层，逐步交换
# 从所有层原始开始，从深层到浅层逐层交换
progressive_results = []

# 按索引排序（1 到 18）
sorted_layers = sorted([idx for idx, _ in decoder_moe])

for swap_count in range(len(sorted_layers) + 1):
    layers_to_swap = set(sorted_layers[:swap_count])   # 先交换浅层
    handles = []

    for idx, moe in decoder_moe:
        if idx in layers_to_swap:
            def mk_swap(ref):
                def hook(m, i, o):
                    with torch.no_grad():
                        inp = i[0]
                        g = ref.gate(inp)
                        flat = inp.view(-1, inp.shape[-1])
                        swapped = m.moe_infer(flat, g[0], g[1]).view_as(inp)
                        if m.shared_experts is not None:
                            swapped = swapped + m.shared_experts(inp)
                    return (swapped, o[1])
                return hook
            handles.append(moe.register_forward_hook(mk_swap(mtp_moe)))

    with torch.no_grad():
        out = model(input_ids=input_ids, use_cache=False, return_dict=True)

    for h in handles:
        h.remove()

    logits = out.logits.float()
    probs = F.softmax(logits, dim=-1)
    tokens = logits.argmax(dim=-1)
    top_conf = probs.max(dim=-1)[0]
    avg_conf = top_conf[0].mean().item()
    unique_tokens = tokens[0].unique().numel()
    most_common = tokens[0].mode().values.item()
    mc_str = tokenizer.decode(most_common)
    local = f"L{sorted_layers[0]}..L{sorted_layers[min(swap_count,len(sorted_layers))-1]}" if swap_count > 0 else "none"
    logit_cos = 0 if swap_count == 0 else F.cosine_similarity(
        logits[0].float().view(-1), outputs.logits[0].float().view(-1), dim=0).item()
    
    match = (tokens == outputs.logits.argmax(dim=-1)).sum().item()
    progressive_results.append((swap_count, unique_tokens, mc_str, avg_conf, logit_cos, match))
    
    print(f"  Swap {swap_count:2d} layers ({layers_to_swap}): uniq={unique_tokens:2d} mode={mc_str:<8} conf={avg_conf:.3f} cos={logit_cos:.4f} match={match}/{T}")

# 测试2：`!` 从哪里来？ Check lm_head weights
print(f"\n{'='*80}")
print("Where does '!' come from? Check lm_head weight norm per token")
print(f"{'='*80}")

lm_w = model.lm_head.weight    # [V, D]
lm_norm = lm_w.norm(dim=1)

# 查找 `!` 的 token id
excl_id = tokenizer.encode("!")[0]
print(f"  '!' token ID: {excl_id}")
print(f"  '!' weight norm: {lm_norm[excl_id].item():.2f}")
print(f"  Mean weight norm: {lm_norm.mean().item():.2f}")
print(f"  Max weight norm: {lm_norm.max().item():.2f}")
print(f"  '!' rank by norm: {(lm_norm > lm_norm[excl_id]).sum().item()}/{lm_norm.shape[0]}")

# 按权重范数排序的前 10 个 token
top10 = lm_norm.topk(10)
print(f"  Top-10 tokens by weight norm:")
for i in range(10):
    tok = tokenizer.decode(top10.indices[i].item())
    print(f"    {top10.indices[i].item():6d} norm={top10.values[i].item():.2f} '{tok}'")

# 测试3：坍缩后所有位置产生相同隐藏状态吗？
print(f"\n{'='*80}")
print("Collapse analysis: Are all positions producing identical hidden states?")
print(f"{'='*80}")

# 注册所有 MoE 层的钩子 + 交换
handles = []
for idx, moe in decoder_moe:
    def mk_swap(ref):
        def hook(m, i, o):
            with torch.no_grad():
                inp = i[0]
                g = ref.gate(inp)
                flat = inp.view(-1, inp.shape[-1])
                swapped = m.moe_infer(flat, g[0], g[1]).view_as(inp)
                if m.shared_experts is not None:
                    swapped = swapped + m.shared_experts(inp)
            return (swapped, o[1])
        return hook
    handles.append(moe.register_forward_hook(mk_swap(mtp_moe)))

# 捕获最终隐藏状态
collapsed_hidden = None
def capture_hidden(m, i, o):
    global collapsed_hidden
    collapsed_hidden = o[0].detach().cpu()

# 也捕获最后一层 Decoder 的输入用于比较
last_layer = model.model.layers[17]   # 第 18 层
h_last = last_layer.register_forward_hook(capture_hidden)

with torch.no_grad():
    out_collapsed = model(input_ids=input_ids, use_cache=False, return_dict=True)

h_last.remove()
for h in handles:
    h.remove()

collapsed_logits = out_collapsed.logits.float()
collapsed_tokens = collapsed_logits.argmax(dim=-1)

# 检查所有位置是否产生相似的隐藏状态
print(f"  After collapse (all layers swapped):")
pos_diffs = []
for t in range(1, T):
    diff = (collapsed_hidden[0, 0] - collapsed_hidden[0, t]).norm().item()
    pos_diffs.append(diff)
print(f"  Mean inter-position L2 diff: {sum(pos_diffs)/len(pos_diffs):.2f}")
print(f"  Max inter-position L2 diff: {max(pos_diffs):.2f}")
print(f"  Min inter-position L2 diff: {min(pos_diffs):.2f}")

# 与正常前向比较
with torch.no_grad():
    normal_out = model(input_ids=input_ids, use_cache=False, return_dict=True)
normal_hidden_path = normal_out.hidden_states
normal_last_hidden = normal_hidden_path[-2]   # 最后一层 Decoder 输出（norm 前）

normal_diffs = []
for t in range(1, T):
    diff = (normal_last_hidden[0, 0] - normal_last_hidden[0, t]).norm().item()
    normal_diffs.append(diff)
print(f"  Normal (no swap) inter-position L2 diff: {sum(normal_diffs)/len(normal_diffs):.2f}")

# 测试4：如果只交换 L1，其余正常？
print(f"\n{'='*80}")
print("Is collapse caused by L1 alone?")
print(f"{'='*80}")

for single_layer in [1, 5, 10, 18]:
    handles = []
    for idx, moe in decoder_moe:
        if idx == single_layer:
            def mk_swap(ref):
                def hook(m, i, o):
                    with torch.no_grad():
                        inp = i[0]
                        g = ref.gate(inp)
                        flat = inp.view(-1, inp.shape[-1])
                        swapped = m.moe_infer(flat, g[0], g[1]).view_as(inp)
                        if m.shared_experts is not None:
                            swapped = swapped + m.shared_experts(inp)
                    return (swapped, o[1])
                return hook
            handles.append(moe.register_forward_hook(mk_swap(mtp_moe)))

    with torch.no_grad():
        out_single = model(input_ids=input_ids, use_cache=False, return_dict=True)

    for h in handles:
        h.remove()

    slogits = out_single.logits.float()
    suniq = slogits.argmax(dim=-1)[0].unique().numel()
    logit_cos_single = F.cosine_similarity(
        slogits[0].float().view(-1), outputs.logits[0].float().view(-1), dim=0).item()
    print(f"  Swap only L{single_layer}: unique_tokens={suniq}, logit_cos={logit_cos_single:.4f}")

print(f"\nDone.")